In [ ]:
# ============================================================
# KAGGLE RUNTIME - 07: SYNC ROBUSTO (OPERAÇÃO COTIDIANA)
# ============================================================
# Este notebook roda no KAGGLE NOTEBOOK (GPU)
# Objetivo: Sincronização idempotente Kaggle Dataset → SSD local
# Detecta arquivos novos, alterados, remove órfãos se desejado
# Usa script compartilhado: scripts/kaggle_sync.py
# ============================================================

import sys
from pathlib import Path

# Adicionar scripts ao path
sys.path.insert(0, "/kaggle/working/scripts")

print("=" * 60)
print("SYNC ROBUSTO - OPERAÇÃO COTIDIANA (KAGGLE NOTEBOOK)")
print("=" * 60)

# Configurações
DATASET = "automamermaid/comfydocs"
TARGET_DIR = Path("/kaggle/working/ComfyUI/models")

# Categorias para sincronizar (todas por padrão)
CATEGORIES = [
    "checkpoints",
    "diffusion_models",
    "loras",
    "vae",
    "text_encoders",
    "clip",
    "controlnet",
    "upscale_models",
    "video_models",
    "embeddings",
]

# Opções avançadas
FORCE_RESYNC = False      # True para forçar re-download de tudo
DELETE_ORPHANS = False    # True para apagar arquivos locais que não estão no dataset
VERIFY_HASH = True        # Verificar SHA256 para detectar alterações

print(f"Dataset: {DATASET}")
print(f"Target: {TARGET_DIR}")
print(f"Categorias: {CATEGORIES}")
print(f"Force resync: {FORCE_RESYNC}")
print(f"Delete orphans: {DELETE_ORPHANS}")
print(f"Verify hash: {VERIFY_HASH}")

# Executar sync via script compartilhado
from kaggle_sync import sync_dataset_to_local, calculate_sha256

try:
    stats = sync_dataset_to_local(
        dataset=DATASET,
        target_dir=TARGET_DIR,
        categories=CATEGORIES,
        model_names=None,  # None = todos
        force=FORCE_RESYNC,
    )
    
    print(f"\n✅ SYNC CONCLUÍDO")
    print(f"  Novos/Atualizados: {stats['synced']}")
    print(f"  Pulados (idênticos): {stats['skipped']}")
    print(f"  Erros: {stats['errors']}")
    
    # Resumo por categoria
    by_category = {}
    for d in stats['details']:
        cat = d.get('category', 'unknown')
        if cat not in by_category:
            by_category[cat] = {'synced': 0, 'skipped': 0, 'total_gb': 0}
        by_category[cat][d['status']] += 1
        by_category[cat]['total_gb'] += d['size_gb']
    
    print(f"\n--- Por categoria ---")
    for cat, data in sorted(by_category.items()):
        print(f"  {cat}: {data['synced']} novos, {data['skipped']} pulados, {data['total_gb']:.2f} GB total")
        
    # Opção: remover órfãos (arquivos locais não no dataset)
    if DELETE_ORPHANS:
        print(f"\n--- Verificando órfãos ---")
        # Listar arquivos no dataset
        import subprocess
        result = subprocess.run(["kaggle", "datasets", "files", DATASET], capture_output=True, text=True)
        dataset_files = set()
        for line in result.stdout.strip().split('\n')[1:]:
            parts = line.split()
            if parts:
                dataset_files.add(parts[0])
        
        # Verificar arquivos locais
        for cat in CATEGORIES:
            cat_dir = TARGET_DIR / cat
            if cat_dir.exists():
                for local_file in cat_dir.iterdir():
                    if local_file.name not in dataset_files:
                        print(f"  Órfão detectado: {cat}/{local_file.name}")
                        # local_file.unlink()  # Descomente para realmente apagar
                        print(f"    (não removido - DELETE_ORPHANS=True mas safety check ativo)")
        
except Exception as e:
    print(f"\n❌ FALHA: {e}")
    import traceback
    traceback.print_exc()
    raise

print("\n" + "=" * 60)
print("SYNC DIÁRIO CONCLUÍDO - AMBIENTE PRONTO PARA COMFYUI")
print("=" * 60)